# Spatial Differential Expression + Gene Set Enrichment Test Dataset

Builds `test-data/visium-brain-de-enrichment-test-data-format.zarr` from the
two-section Visium mouse brain SpatialData store produced by
`transform_visium_brain.py`.

The output is a full SpatialData store (both images, both shapes elements, one
table) whose table additionally carries:

- **`uns/de`** — differential expression contrasts following
  `DGE Zarr Specification.md`.
- **`uns/enrichment`** — gene set enrichment following
  `Enrichment Zarr Specification.md`, computed with
  [decoupler](https://decoupler-py.readthedocs.io/) 2.x.
- **`obsm/X_activity_*`** — per-spot gene set activity matrices, which render
  directly on the tissue.

Three contrast series are produced:

| Series | Contrast column | Type | Subset |
| --- | --- | --- | --- |
| 1 | `region` | pairwise, `ST8059048` vs `ST8059050` | none |
| 2 | `cluster` | one-vs-rest per leiden cluster | none |
| 3 | `cluster` | one-vs-rest per leiden cluster | `region = ST8059048` |

Series 1 is the slide-vs-slide comparison and is the only contrast in the repo
that exercises the pairwise (`group_2` non-null) path. Series 3 exercises the
subset-restricted path, mirroring the hippocampus series of the habib17 store.

Run `transform_visium_brain.py` first. Requires network access for decoupler's
`dc.op` gene set downloads (OmniPath / MSigDB).

In [1]:
from pathlib import Path
import json
import shutil
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Optional
import importlib.metadata

In [2]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
import decoupler as dc

np.random.seed(1)
sc.settings.verbosity = 1

/home/klaus/ws/zarr-test-datasets/.claude/worktrees/enrichment-subset-comparisons-9a20d0/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [3]:
# The pristine SpatialData store written by transform_visium_brain.py.
SOURCE_STORE = Path("test-data/visium_brain.spatialdata.zarr")
OUTPUT_PATH = Path("test-data/visium-brain-de-enrichment-test-data-format.zarr")
# Every DE / enrichment path below is relative to the AnnData node inside the
# SpatialData store, not to the store root.
TABLE_KEY = "table"

# Mouse brain sections -> mouse gene symbols -> mouse gene set collections.
ORGANISM = "mouse"

# obs columns. `region` already exists in the source table and holds the two
# section ids; `cluster` is computed below.
SLIDE_COLUMN = "region"
CLUSTER_COLUMN = "cluster"
# The two serial sections. Series 1 compares them directly; series 3 restricts
# the per-cluster contrasts to the first one.
SLIDE_A = "ST8059048"
SLIDE_B = "ST8059050"

# Preprocessing / clustering.
MIN_CELLS_PER_GENE = 10   # drop genes detected in fewer spots than this
N_TOP_GENES = 2000        # highly variable genes feeding the PCA
LEIDEN_RESOLUTION = 0.5
RANDOM_STATE = 0

MIN_CELLS = 5             # minimum spots in group_1 to run a contrast
SIGNIFICANCE_GAP_FACTOR = 1.5  # zero/underflowed p -> cap = ceil(max_finite * this)

# Enrichment settings, matching calculate-gene-set-enrichment-data.ipynb.
ORA_N_UP = 50   # top up-regulated genes ORA treats as the observed signature
TMIN = 5        # minimum genes per set present in the data (decoupler `tmin`)
# Contrasts scored against the large collections, keeping the build bounded.
N_SMALL_COLLECTION_CONTRASTS = 5

# decoupler >= 2 returns a single p-value matrix per testing method and, for
# every method EXCEPT these, overwrites it in place with the BH-adjusted values
# before returning (see decoupler/mt/_run.py: `if name != "mlm"`). For methods
# listed here the returned p-values are RAW (unadjusted), so we must store them
# as `pvals` (not `pvals_adj`) and label significance accordingly.
UNADJUSTED_PVAL_METHODS = {"mlm"}

## Preprocessing and clustering

The source table is **raw counts** (mouse, 6484 spots x 31053 genes) and has no
cell type or cluster annotation — only the two section ids in `obs["region"]`.
Both scanpy's DE and decoupler need log-normalized input, so counts are kept in
`layers/counts` and `X` becomes the normalized matrix. Leiden clusters then give
the grouping the one-vs-rest series need.

In [4]:
def preprocess(adata: ad.AnnData) -> ad.AnnData:
    """Counts -> layers/counts, X -> normalized + log1p, low-detection genes dropped."""
    adata.var_names_make_unique()
    adata.layers["counts"] = adata.X.copy()
    n_before = adata.n_vars
    sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
    print(f"Kept {adata.n_vars}/{n_before} genes detected in >= {MIN_CELLS_PER_GENE} spots")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("Applied preprocessing: normalize_total(target_sum=1e4) + log1p")
    return adata


def add_clusters(adata: ad.AnnData) -> ad.AnnData:
    """Leiden clusters on the PCA of the highly variable genes, plus a UMAP.

    `sc.pp.highly_variable_genes` only flags genes, it does not subset them, so
    `use_highly_variable=True` on the PCA is what restricts the embedding. The
    UMAP is written so the store also works in the app's non-spatial views.
    """
    sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES)
    sc.pp.pca(adata, n_comps=50, use_highly_variable=True, random_state=RANDOM_STATE)
    sc.pp.neighbors(adata, n_neighbors=15, random_state=RANDOM_STATE)
    sc.tl.leiden(
        adata,
        resolution=LEIDEN_RESOLUTION,
        key_added=CLUSTER_COLUMN,
        flavor="igraph",
        n_iterations=2,
        directed=False,
        random_state=RANDOM_STATE,
    )
    # Readable, stably ordered category labels ("cluster_0", "cluster_1", ...).
    codes = adata.obs[CLUSTER_COLUMN].astype(int)
    labels = [f"cluster_{c}" for c in codes]
    adata.obs[CLUSTER_COLUMN] = pd.Categorical(
        labels, categories=[f"cluster_{c}" for c in sorted(set(codes))]
    )
    sc.tl.umap(adata, random_state=RANDOM_STATE)
    counts = adata.obs[CLUSTER_COLUMN].value_counts().sort_index()
    print(f"Found {len(counts)} leiden clusters: {counts.to_dict()}")
    return adata

## DE contrasts -> `tables/table/uns/de/`

Same 11 arrays and the same pre-sort as the habib17 DE store, so the app reads
both identically.

`pct_expr_target` / `pct_expr_ref` are computed from the group masks rather than
taken from scanpy's `pts` / `pts_rest`: scanpy only fills `pts_rest` when the
reference is `"rest"`, so the pairwise slide-vs-slide contrast would otherwise
have no reference percentage.

In [5]:
@dataclass
class ContrastConfig:
    """Configuration for a single DE contrast."""
    contrast_id: str
    contrast_column: str        # obs column defining the groups
    group_1: str                # target group
    method: str
    corr_method: Optional[str] = None
    test_type: str = "one_vs_rest"
    group_2: Optional[str] = None       # reference group; None -> rest
    subset_column: Optional[str] = None  # obs column restricting the spots
    subset_value: Optional[str] = None


def build_contrast_matrix(clusters: list[str], slide_a_clusters: list[str]) -> list[ContrastConfig]:
    """Three deterministic series; ids run de_001.. across all of them."""
    contrasts: list[ContrastConfig] = []

    # Series 1: the two tissue slides against each other (pairwise).
    contrasts.append(ContrastConfig(
        contrast_id="de_001",
        contrast_column=SLIDE_COLUMN,
        group_1=SLIDE_A,
        group_2=SLIDE_B,
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        test_type="pairwise",
    ))

    # Series 2: one-vs-rest per cluster over both slides.
    for i, cl in enumerate(clusters, len(contrasts) + 1):
        contrasts.append(ContrastConfig(
            contrast_id=f"de_{i:03d}",
            contrast_column=CLUSTER_COLUMN,
            group_1=cl,
            method="wilcoxon",
            corr_method="benjamini-hochberg",
            test_type="one_vs_rest",
        ))

    # Series 3: the same one-vs-rest contrasts restricted to one slide.
    for i, cl in enumerate(slide_a_clusters, len(contrasts) + 1):
        contrasts.append(ContrastConfig(
            contrast_id=f"de_{i:03d}",
            contrast_column=CLUSTER_COLUMN,
            group_1=cl,
            method="wilcoxon",
            corr_method="benjamini-hochberg",
            test_type="one_vs_rest",
            subset_column=SLIDE_COLUMN,
            subset_value=SLIDE_A,
        ))

    return contrasts

In [6]:
def _group_stats(adata: ad.AnnData, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """(mean expression, percent of spots expressing) per gene, in var order."""
    n = int(mask.sum())
    if n == 0:
        nan = np.full(adata.n_vars, np.nan)
        return nan, nan
    X = adata.X[mask]
    mean = np.asarray(X.mean(axis=0)).ravel()
    # (X > 0) works for both sparse and dense; .sum(axis=0) counts expressing spots.
    n_expressing = np.asarray((X > 0).sum(axis=0)).ravel()
    return mean, 100.0 * n_expressing / n


def _field_for_group(result: dict, key: str, group: str, n_genes: int) -> np.ndarray:
    """Pull one rank_genes_groups column, tolerating DataFrame or structured array."""
    if key not in result:
        return np.full(n_genes, np.nan)
    field = result[key]
    if hasattr(field, "columns"):  # DataFrame
        if group not in field.columns:
            return np.full(n_genes, np.nan)
        return np.asarray(field[group].values)
    if field.dtype.names is None or group not in field.dtype.names:  # structured array
        return np.full(n_genes, np.nan)
    return np.asarray(field[group])


def extract_contrast_arrays(
    adata: ad.AnnData,
    contrast_key: str,
    config: ContrastConfig,
    mean_expr_global: np.ndarray,
) -> dict:
    """Build the 11 spec arrays for one contrast, ordered by rank_genes_groups."""
    result = adata.uns[contrast_key]
    group_1 = config.group_1
    names_field = result["names"]
    gene_id = np.asarray(
        names_field[group_1].values if hasattr(names_field, "columns") else names_field[group_1]
    ).astype(str)
    n_genes = len(gene_id)

    scores = _field_for_group(result, "scores", group_1, n_genes)
    effect_size = _field_for_group(result, "logfoldchanges", group_1, n_genes)
    pvals = _field_for_group(result, "pvals", group_1, n_genes)
    pvals_adj = _field_for_group(result, "pvals_adj", group_1, n_genes)

    # Expression summaries are computed per gene in var order, then reindexed to
    # the rank_genes_groups gene order.
    groups = adata.obs[config.contrast_column].astype(str).to_numpy()
    target_mask = groups == group_1
    ref_mask = (groups == config.group_2) if config.group_2 is not None else ~target_mask
    mean_target, pct_target = _group_stats(adata, target_mask)
    mean_ref, pct_ref = _group_stats(adata, ref_mask)

    var_order = pd.Index(adata.var_names.astype(str)).get_indexer(gene_id)
    if (var_order < 0).any():
        raise KeyError(f"{contrast_key}: rank_genes_groups returned genes absent from var_names")

    return {
        "gene_id": gene_id,
        "effect_size": effect_size,
        "pvals": pvals,
        "pvals_adj": pvals_adj,
        "significance": _compute_significance(pvals_adj)[0],
        "scores": scores,
        "pct_expr_target": pct_target[var_order],
        "pct_expr_ref": pct_ref[var_order],
        "mean_expr": mean_expr_global[var_order],
        "mean_expr_target": mean_target[var_order],
        "mean_expr_ref": mean_ref[var_order],
    }


def presort_contrast_arrays(arrays: dict) -> dict:
    """Descending scores, then ascending pvals_adj, then ascending gene id."""
    sort_idx = np.lexsort((
        arrays["gene_id"],
        np.where(np.isnan(arrays["pvals_adj"]), np.inf, arrays["pvals_adj"]),
        -arrays["scores"],
    ))
    return {key: arr[sort_idx] for key, arr in arrays.items()}

## Shared numeric helpers

`significance` is the plot-ready `-log10(p)` the app reads directly, with a
data-driven cap so underflowed p-values stay on the chart. Identical to the
habib17 notebooks.

In [7]:
def _compute_significance(
    pvals: np.ndarray,
    gap_factor: float = SIGNIFICANCE_GAP_FACTOR,
) -> tuple[np.ndarray, Optional[float]]:
    """Plot-ready -log10(p) with a data-driven cap.

    NaN inputs stay NaN. Zero/underflowed p-values (infinite -log10) are pinned
    to cap = ceil(max_finite * gap_factor), leaving a readable gap above the
    largest real value; the cap always equals np.nanmax(significance).
    """
    padj = np.asarray(pvals, dtype=float)
    significance = np.full(padj.shape, np.nan)
    valid = ~np.isnan(padj)
    if not np.any(valid):
        return significance, None
    with np.errstate(divide="ignore"):
        logp = -np.log10(padj[valid])  # p == 0 -> inf
    finite = logp[np.isfinite(logp)]
    if len(finite) == 0:
        cap = 300.0  # degenerate: every valid p underflowed to 0
    elif np.any(np.isinf(logp)):
        cap = float(np.ceil(np.max(finite) * gap_factor))
    else:
        cap = float(np.ceil(np.max(finite)))
    significance[valid] = np.minimum(np.where(np.isinf(logp), cap, logp), cap)
    return significance, cap


def _effect_size_max(effect_size: np.ndarray) -> Optional[float]:
    """Symmetric x-axis bound: ceil of the largest absolute finite effect size."""
    valid = effect_size[~(np.isnan(effect_size) | np.isinf(effect_size))]
    return float(np.ceil(np.max(np.abs(valid)))) if len(valid) > 0 else None


def _significance_max(significance: np.ndarray) -> Optional[float]:
    """Y-axis bound, read off the precomputed array so bound and array agree."""
    return float(np.nanmax(significance)) if np.any(~np.isnan(significance)) else None

## DE registry entries

The arrays themselves go into `adata.uns["de"]` as a nested dict, which AnnData
writes as the `uns/de/<contrast_id>/<array>` layout the spec asks for. Only
`contrast_registry.json` is written by hand afterwards, because it is a plain
JSON file rather than a Zarr node.

In [8]:
def build_contrast_metadata(arrays: dict, config: ContrastConfig) -> dict:
    """One `contrasts[]` entry of contrast_registry.json."""
    effect_size_max = _effect_size_max(arrays["effect_size"])
    significance_max = _significance_max(arrays["significance"])

    has_effect_size = effect_size_max is not None
    has_significance = significance_max is not None
    correction_method = config.corr_method if has_significance else None
    if has_significance and correction_method is None:
        correction_method = "uncorrected"

    return {
        "contrast_id": config.contrast_id,
        "group_1": config.group_1,
        "group_2": config.group_2,
        "test_type": config.test_type,
        "subset_column": config.subset_column,
        "subset_value": config.subset_value,
        "contrast_column": config.contrast_column,
        "de_method": config.method,
        "correction_method": correction_method,
        "has_effect_size": has_effect_size,
        "has_significance": has_significance,
        "has_pct_expr_target": bool(np.any(~np.isnan(arrays["pct_expr_target"]))),
        "has_pct_expr_ref": bool(np.any(~np.isnan(arrays["pct_expr_ref"]))),
        "has_mean_expr_target": bool(np.any(~np.isnan(arrays["mean_expr_target"]))),
        "has_mean_expr_ref": bool(np.any(~np.isnan(arrays["mean_expr_ref"]))),
        "feature_type": "Gene Expression",
        "effect_size_label": "log2(Fold Change)" if has_effect_size else None,
        "significance_label": (
            ("-log10(FDR)" if config.corr_method else "-log10(p)") if has_significance else None
        ),
        "effect_size_max": effect_size_max,
        "significance_max": significance_max,
        "top_10_gene_ids": arrays["gene_id"].tolist()[:10],
    }


def run_contrasts(adata: ad.AnnData, contrasts: list[ContrastConfig]) -> tuple[dict, list[dict]]:
    """Run every contrast; returns ({contrast_id: arrays}, registry entries)."""
    # Subset views are materialized once and reused by every series-3 contrast.
    subsets: dict[tuple, ad.AnnData] = {}
    means: dict[tuple, np.ndarray] = {}

    def active(config: ContrastConfig) -> tuple[ad.AnnData, np.ndarray]:
        key = (config.subset_column, config.subset_value)
        if key not in subsets:
            if config.subset_column is None:
                sub = adata
            else:
                sub = adata[adata.obs[config.subset_column].astype(str) == config.subset_value].copy()
                # Re-categorize so rank_genes_groups does not see empty groups.
                sub.obs[CLUSTER_COLUMN] = sub.obs[CLUSTER_COLUMN].astype(str).astype("category")
            subsets[key] = sub
            means[key] = np.asarray(sub.X.mean(axis=0)).ravel()
        return subsets[key], means[key]

    de_arrays: dict[str, dict] = {}
    registry: list[dict] = []
    for config in contrasts:
        try:
            sub, mean_expr = active(config)
            group_size = int((sub.obs[config.contrast_column].astype(str) == config.group_1).sum())
            if group_size < MIN_CELLS:
                print(f"  ⊘ {config.contrast_id}: group '{config.group_1}' has "
                      f"{group_size} spots < {MIN_CELLS}")
                continue

            key_added = f"_contrast_{config.contrast_id}"
            sc.tl.rank_genes_groups(
                sub,
                groupby=config.contrast_column,
                groups=[config.group_1],
                reference=config.group_2 if config.group_2 is not None else "rest",
                method=config.method,
                corr_method=config.corr_method,
                tie_correct=True,
                pts=True,
                key_added=key_added,
                use_raw=False,
            )
            arrays = presort_contrast_arrays(
                extract_contrast_arrays(sub, key_added, config, mean_expr)
            )
            del sub.uns[key_added]

            de_arrays[config.contrast_id] = arrays
            registry.append(build_contrast_metadata(arrays, config))
            label = (f"{config.subset_column}={config.subset_value} " if config.subset_column else "")
            print(f"  ✓ {config.contrast_id}: {label}{config.group_1} vs "
                  f"{config.group_2 or 'rest'} ({group_size} spots)")
        except Exception as exc:  # noqa: BLE001 - skip one contrast, not the run
            print(f"  ✗ {config.contrast_id}: {exc}")
    return de_arrays, registry

## Gene set collections (decoupler `dc.op`)

Fetched for mouse. Each collection is a long-format `net` (`source`, `target`,
optional `weight`). A single unreachable resource is skipped with a log line
rather than aborting the run.

In [9]:
def _load_gobp() -> pd.DataFrame:
    """MSigDB C5 GO:Biological Process subcollection as an unweighted net."""
    msig = dc.op.resource("MSigDB", organism=ORGANISM)
    gobp = msig[msig["collection"] == "go_biological_process"]
    return gobp.rename(columns={"geneset": "source", "genesymbol": "target"})[
        ["source", "target"]
    ]


def _load_omnipath() -> pd.DataFrame:
    """OmniPath annotations (Wang subcellular locations) as unweighted gene sets.

    Wang is a human annotation resource, so it is fetched for human and mapped to
    mouse orthologs with dc.op.translate.
    """
    wang = dc.op.resource("Wang", organism="human")
    net = wang.dropna(subset=["location"]).rename(
        columns={"location": "source", "genesymbol": "target"}
    )[["source", "target"]]
    net = net[net["source"].astype(str).str.len() > 0]
    if ORGANISM != "human":
        # Only `target` holds gene symbols here; `source` is a location label.
        net = dc.op.translate(net, target_organism=ORGANISM, columns="target")
    return net


def load_collections() -> dict:
    """Download every gene set collection, skipping any that fail."""
    collections: dict = {}

    def add(key: str, label: str, loader) -> None:
        try:
            net = loader().dropna(subset=["source", "target"]).copy()
            net["source"] = net["source"].astype(str)
            net["target"] = net["target"].astype(str)
            net = net.drop_duplicates(subset=["source", "target"])
            collections[key] = {
                "net": net,
                "label": label,
                "weighted": "weight" in net.columns,
                "set_size": net.groupby("source")["target"].nunique(),
            }
            print(f"  ✓ {label}: {net['source'].nunique()} sets, {len(net)} edges")
        except Exception as exc:  # noqa: BLE001 - want to skip, not abort
            print(f"  ✗ {label}: {exc}")

    add("hallmark", "MSigDB Hallmark",
        lambda: dc.op.hallmark(organism=ORGANISM)[["source", "target"]])
    add("gobp", "GO:BP", _load_gobp)
    add("omnipath", "OmniPath", _load_omnipath)
    add("progeny", "PROGENy",
        lambda: dc.op.progeny(organism=ORGANISM)[["source", "target", "weight"]])
    add("collectri", "CollecTRI",
        lambda: dc.op.collectri(organism=ORGANISM)[["source", "target", "weight"]])
    return collections

## Contrast-based enrichment -> `tables/table/uns/enrichment/es_XXX`

The method x collection matrix. Each (contrast, method, collection) triple is one
`es_XXX` entry. `signed` controls whether a signed effect size is exposed
(GSEA/MLM/ULM yes; ORA reports an over-representation statistic only).

The signature matrix is contrasts x genes, filled with the per-gene DE `scores`
(genes absent from a contrast filled with 0).

In [10]:
@dataclass
class EnrichJob:
    method: str            # decoupler dc.mt.* method name
    collection: str        # collection key
    contrasts: list[str]   # DE contrast ids to run
    signed: bool           # whether a signed effect size is exposed
    effect_size_label: Optional[str]
    input_statistic: str


def build_jobs(all_contrasts: list[str]) -> list[EnrichJob]:
    """Hallmark covers every contrast; the larger collections cover a slice."""
    small = all_contrasts[:N_SMALL_COLLECTION_CONTRASTS]
    return [
        EnrichJob("gsea", "hallmark", all_contrasts, True,
                  "Normalized Enrichment Score", "de_scores"),
        EnrichJob("ora", "hallmark", all_contrasts, False,
                  None, f"top_{ORA_N_UP}_up_genes"),
        EnrichJob("gsea", "gobp", small, True,
                  "Normalized Enrichment Score", "de_scores"),
        EnrichJob("ora", "gobp", small, False,
                  None, f"top_{ORA_N_UP}_up_genes"),
        EnrichJob("ora", "omnipath", small, False,
                  None, f"top_{ORA_N_UP}_up_genes"),
        EnrichJob("mlm", "progeny", small, True,
                  "MLM t-value", "de_scores"),
        EnrichJob("ulm", "collectri", small, True,
                  "ULM t-value", "de_scores"),
    ]


def build_signature(de_arrays: dict, var_names: np.ndarray) -> pd.DataFrame:
    """contrasts x genes matrix of DE scores, reindexed to the full gene axis."""
    rows: dict[str, pd.Series] = {}
    for cid, arrays in de_arrays.items():
        series = pd.Series(
            np.asarray(arrays["scores"], dtype=float),
            index=pd.Index(arrays["gene_id"]).astype(str),
        )
        series = series[~series.index.duplicated(keep="first")]
        rows[cid] = series.reindex(var_names).fillna(0.0)
    mat = pd.DataFrame(rows).T
    mat.columns = var_names
    return mat


def run_method(method: str, mat: pd.DataFrame, net: pd.DataFrame):
    """Run a decoupler method; returns (score_df, pv_df) shaped rows x sources.

    `pv_df` is the p-value matrix decoupler returns for testing methods (None for
    non-testing ones). For every method except those in UNADJUSTED_PVAL_METHODS
    the returned values are already BH-adjusted; for those (e.g. mlm) they are
    raw. The caller decides which array (`pvals` vs `pvals_adj`) they belong in.
    """
    fn = getattr(dc.mt, method)
    kwargs = {"tmin": TMIN}
    if method == "ora":
        kwargs["n_up"] = ORA_N_UP
    out = fn(mat, net, verbose=False, **kwargs)
    score, pv = out if isinstance(out, tuple) else (out, None)
    return score, pv


def overlap_sizes(net: pd.DataFrame, var_names: np.ndarray) -> pd.Series:
    """Genes per set that are present/tested in the data."""
    present = net[net["target"].isin(set(map(str, var_names)))]
    return present.groupby("source")["target"].nunique()

In [11]:
def presort_enrichment_arrays(arrays: dict) -> dict:
    """Sort every array descending by scores (NaN last), stable tie-breakers.

    The p-value tie-breaker uses whichever of pvals_adj/pvals is populated
    (adjusted preferred), so methods that expose only raw p-values (e.g. mlm)
    still get a meaningful secondary ordering.
    """
    scores = arrays["scores"]
    pval_tiebreak = np.where(
        np.isnan(arrays["pvals_adj"]), arrays["pvals"], arrays["pvals_adj"]
    )
    sort_idx = np.lexsort((
        arrays["gene_set_id"],                                      # ascending id
        np.where(np.isnan(pval_tiebreak), np.inf, pval_tiebreak),   # ascending p
        -np.where(np.isnan(scores), -np.inf, scores),               # descending scores
    ))
    return {key: arr[sort_idx] for key, arr in arrays.items()}


def assemble_enrichment_arrays(cid, score_df, pv_df, set_size, ov, signed,
                               pvals_are_adjusted=True) -> dict:
    """Build the stable per-entry array set for one contrast row.

    `pv_df` holds the p-values decoupler returned. When `pvals_are_adjusted` is
    True they land in `pvals_adj` (BH-corrected); otherwise they are raw and land
    in `pvals`. Significance (-log10 p) is derived from whichever array is filled.
    """
    sources = np.asarray(score_df.columns).astype(str)
    scores = np.asarray(score_df.loc[cid].to_numpy(), dtype=float)
    pvals = np.full(len(sources), np.nan)
    pvals_adj = np.full(len(sources), np.nan)
    if pv_df is not None:
        pv = np.asarray(pv_df.loc[cid].to_numpy(), dtype=float)
        if pvals_are_adjusted:
            pvals_adj = pv
        else:
            pvals = pv
    significance_source = pvals_adj if pvals_are_adjusted else pvals
    return {
        "gene_set_id": sources,
        "scores": scores,
        "effect_size": scores.copy() if signed else np.full(len(sources), np.nan),
        "pvals": pvals,
        "pvals_adj": pvals_adj,
        "significance": _compute_significance(significance_source)[0],
        "set_size": set_size.reindex(sources).to_numpy(dtype=float),
        "overlap_size": ov.reindex(sources).fillna(0).to_numpy(dtype=float),
    }


def build_enrichment_metadata(es_id, cid, arrays, job, collection_label, de_meta,
                              pvals_are_adjusted=True) -> dict:
    """One `contrast_enrichments[]` entry of enrichment_registry.json."""
    effect_size_max = _effect_size_max(arrays["effect_size"])
    significance_max = _significance_max(arrays["significance"])
    has_effect_size = bool(job.signed and effect_size_max is not None)
    has_significance = significance_max is not None

    if not has_significance:
        correction_method = None
        significance_label = None
    elif pvals_are_adjusted:
        correction_method = "benjamini-hochberg"
        significance_label = "-log10(FDR)"
    else:
        correction_method = None  # decoupler does not FDR-adjust this method (e.g. mlm)
        significance_label = "-log10(p-value)"

    return {
        "enrichment_id": es_id,
        "source_contrast_id": cid,
        "group_1": de_meta["group_1"],
        "group_2": de_meta["group_2"],
        "test_type": de_meta["test_type"],
        "subset_column": de_meta["subset_column"],
        "subset_value": de_meta["subset_value"],
        "contrast_column": de_meta["contrast_column"],
        "enrichment_method": job.method,
        "gene_set_collection": collection_label,
        "n_gene_sets": int(len(arrays["gene_set_id"])),
        "input_statistic": job.input_statistic,
        "has_effect_size": has_effect_size,
        "has_significance": has_significance,
        "has_set_size": bool(np.any(~np.isnan(arrays["set_size"]))),
        "has_overlap": bool(np.any(~np.isnan(arrays["overlap_size"]))),
        "has_leading_edge": False,
        "correction_method": correction_method,
        "effect_size_label": job.effect_size_label if has_effect_size else None,
        "significance_label": significance_label,
        "effect_size_max": effect_size_max if has_effect_size else None,
        "significance_max": significance_max if has_significance else None,
        "top_10_gene_set_ids": arrays["gene_set_id"].tolist()[:10],
    }


def run_enrichment(signature, jobs, collections, de_by_id, var_names) -> tuple[dict, list[dict]]:
    """Run every job; returns ({enrichment_id: arrays}, registry entries)."""
    es_arrays: dict[str, dict] = {}
    registry: list[dict] = []
    es_counter = 1
    for job in jobs:
        if job.collection not in collections:
            print(f"  ✗ {job.method} x {job.collection}: collection unavailable, skipped")
            continue
        collection = collections[job.collection]
        contrasts = [c for c in job.contrasts if c in de_by_id and c in signature.index]
        try:
            score_df, pv_df = run_method(job.method, signature.loc[contrasts], collection["net"])
        except Exception as exc:  # noqa: BLE001
            print(f"  ✗ {job.method} x {job.collection}: {exc}")
            continue
        # decoupler skips FDR correction for some methods (e.g. mlm); their pv is raw.
        pvals_are_adjusted = job.method not in UNADJUSTED_PVAL_METHODS
        ov = overlap_sizes(collection["net"], var_names)
        for cid in contrasts:
            if cid not in score_df.index:
                print(f"    ⊘ {cid}: no result for {job.method} x {job.collection}")
                continue
            es_id = f"es_{es_counter:03d}"
            arrays = presort_enrichment_arrays(assemble_enrichment_arrays(
                cid, score_df, pv_df, collection["set_size"], ov, job.signed,
                pvals_are_adjusted,
            ))
            es_arrays[es_id] = arrays
            registry.append(build_enrichment_metadata(
                es_id, cid, arrays, job, collection["label"], de_by_id[cid],
                pvals_are_adjusted,
            ))
            es_counter += 1
        print(f"  ✓ {job.method} x {collection['label']}: {len(contrasts)} contrasts")
    return es_arrays, registry

## Per-spot activity -> `tables/table/obsm/`

A few methods run on the full expression matrix, producing spots x gene sets
activity matrices. Each becomes an `obsm/X_activity_<method>_<collection>` array
(column order recorded as `gene_set_ids` in the registry). On a spatial dataset
these are the overlays that render on the tissue.

In [12]:
CELL_ACTIVITY_COMBOS = [
    ("mlm", "progeny", "MLM t-value"),
    ("ulm", "collectri", "ULM t-value"),
    ("aucell", "hallmark", "AUCell enrichment score"),
]


def run_spot_activities(adata, collections) -> list[dict]:
    """Compute per-spot activity matrices; returns metadata + score DataFrames."""
    X = adata.X
    dense = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    expr = pd.DataFrame(
        dense,
        index=adata.obs_names.astype(str),
        columns=adata.var_names.astype(str),
    )
    activities: list[dict] = []
    for method, key, label in CELL_ACTIVITY_COMBOS:
        if key not in collections:
            print(f"  ✗ activity {method} x {key}: collection unavailable")
            continue
        score, _pv = run_method(method, expr, collections[key]["net"])
        obsm_key = f"X_activity_{method}_{key}"
        activities.append({
            "obsm_key": obsm_key,
            "method": method,
            "collection_key": key,
            "gene_set_collection": collections[key]["label"],
            "value_label": label,
            "gene_set_ids": [str(c) for c in score.columns],
            "score": score,
        })
        print(f"  ✓ activity {obsm_key}: {score.shape[0]} spots x {score.shape[1]} sets")
    return activities

## Build the store

Unlike the habib17 notebooks, nothing here is written into Zarr by hand. The DE
and enrichment arrays are attached to `adata.uns` as nested dicts of NumPy
arrays, and AnnData writes them as the `uns/de/<id>/<array>` and
`uns/enrichment/<id>/<array>` layout the specs describe, with the right
`encoding-type` attributes and inside SpatialData's own consolidated metadata.
That avoids re-consolidating a SpatialData store after the fact, which would
risk breaking `sd.read_zarr`.

The two registries are still written afterwards: they are plain JSON files
fetched by path, not Zarr nodes.

In [13]:
def write_registry(path: Path, payload: dict) -> None:
    """Registries are plain JSON files inside the store, not Zarr nodes."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as fh:
        json.dump(payload, fh, indent=2)


def main() -> None:
    if not SOURCE_STORE.exists():
        raise FileNotFoundError(
            f"Source store not found: {SOURCE_STORE}. Run transform_visium_brain.py first."
        )

    # 1. Read the SpatialData store and take its table.
    sdata = sd.read_zarr(SOURCE_STORE)
    adata = sdata.tables[TABLE_KEY]
    print(f"Source table: {adata.n_obs} spots x {adata.n_vars} genes")
    print(f"Sections: {adata.obs[SLIDE_COLUMN].value_counts().to_dict()}")

    # 2. Preprocess and cluster.
    adata = preprocess(adata)
    adata = add_clusters(adata)
    var_names = adata.var_names.astype(str).to_numpy()

    # 3. Contrast matrix. Series 3 only covers clusters that survive on slide A.
    clusters = adata.obs[CLUSTER_COLUMN].cat.categories.astype(str).tolist()
    slide_a_obs = adata.obs.loc[adata.obs[SLIDE_COLUMN].astype(str) == SLIDE_A, CLUSTER_COLUMN]
    slide_a_counts = slide_a_obs.astype(str).value_counts()
    slide_a_clusters = [c for c in clusters if slide_a_counts.get(c, 0) >= MIN_CELLS]
    contrasts = build_contrast_matrix(clusters, slide_a_clusters)
    print(f"Built contrast matrix with {len(contrasts)} contrasts "
          f"(1 pairwise + {len(clusters)} global + {len(slide_a_clusters)} on {SLIDE_A})")

    print("Running differential expression...")
    de_arrays, de_registry = run_contrasts(adata, contrasts)
    de_by_id = {entry["contrast_id"]: entry for entry in de_registry}

    # 4. Gene set collections.
    print(f"Downloading gene set collections (organism={ORGANISM})...")
    collections = load_collections()

    # 5. Contrast-based enrichment.
    print("Running contrast-based enrichment...")
    signature = build_signature(de_arrays, var_names)
    jobs = build_jobs(list(de_arrays.keys()))
    es_arrays, es_registry = run_enrichment(signature, jobs, collections, de_by_id, var_names)

    # 6. Per-spot activity matrices.
    print("Running per-spot activities...")
    activities = run_spot_activities(adata, collections)
    cell_activities = []
    for act in activities:
        adata.obsm[act["obsm_key"]] = act["score"].to_numpy().astype("float32")
        cell_activities.append({
            "obsm_key": act["obsm_key"],
            "method": act["method"],
            "gene_set_collection": act["gene_set_collection"],
            "value_label": act["value_label"],
            "gene_set_ids": act["gene_set_ids"],
            "n_gene_sets": len(act["gene_set_ids"]),
        })

    # 7. Attach the arrays and write the whole SpatialData store.
    adata.uns["de"] = de_arrays
    adata.uns["enrichment"] = es_arrays
    sdata.tables[TABLE_KEY] = adata
    if OUTPUT_PATH.exists():
        shutil.rmtree(OUTPUT_PATH)
    sdata.write(OUTPUT_PATH)
    print(f"Wrote SpatialData store -> {OUTPUT_PATH}")

    # 8. Registries.
    uns_dir = OUTPUT_PATH / "tables" / TABLE_KEY / "uns"
    write_registry(uns_dir / "de" / "contrast_registry.json", {
        "version": "1.0",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "scanpy_version": importlib.metadata.version("scanpy"),
        "zarr_version": importlib.metadata.version("zarr"),
        "dataset_metadata": {
            "n_obs": int(adata.n_obs),
            "n_vars": int(adata.n_vars),
            "preprocessing": "normalize_total(target_sum=1e4) + log1p",
            "source_store": SOURCE_STORE.name,
        },
        "contrast_summary": {
            "total_contrasts": len(de_registry),
            "series": [
                {
                    "series_id": 1,
                    "method": "wilcoxon",
                    "corr": "benjamini-hochberg",
                    "contrast_column": SLIDE_COLUMN,
                    "n_contrasts": 1,
                },
                {
                    "series_id": 2,
                    "method": "wilcoxon",
                    "corr": "benjamini-hochberg",
                    "contrast_column": CLUSTER_COLUMN,
                    "n_contrasts": len(clusters),
                },
                {
                    "series_id": 3,
                    "method": "wilcoxon",
                    "corr": "benjamini-hochberg",
                    "contrast_column": CLUSTER_COLUMN,
                    "subset_column": SLIDE_COLUMN,
                    "subset_value": SLIDE_A,
                    "n_contrasts": len(slide_a_clusters),
                },
            ],
        },
        "contrasts": de_registry,
    })

    write_registry(uns_dir / "enrichment" / "enrichment_registry.json", {
        "version": "1.0",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "decoupler_version": importlib.metadata.version("decoupler"),
        "zarr_version": importlib.metadata.version("zarr"),
        "dataset_metadata": {
            "n_obs": int(adata.n_obs),
            "n_vars": int(adata.n_vars),
            "source_store": SOURCE_STORE.name,
        },
        "enrichment_summary": {
            "total_enrichments": len(es_registry),
            "total_cell_activities": len(cell_activities),
            "series": [
                {
                    "method": job.method,
                    "gene_set_collection": collections[job.collection]["label"]
                    if job.collection in collections else job.collection,
                    "n_contrasts": len(job.contrasts),
                    "available": job.collection in collections,
                }
                for job in jobs
            ],
        },
        "contrast_enrichments": es_registry,
        "cell_activities": cell_activities,
    })

    print(f"\nWrote {len(de_registry)} DE contrasts, {len(es_registry)} enrichments "
          f"and {len(cell_activities)} activity matrices to {OUTPUT_PATH}")

In [14]:
if __name__ == "__main__":
    main()

Source table: 6484 spots x 31053 genes
Sections: {'ST8059050': 3497, 'ST8059048': 2987}


Kept 18483/31053 genes detected in >= 10 spots


Applied preprocessing: normalize_total(target_sum=1e4) + log1p


/tmp/ipykernel_1413691/4179496869.py:22: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  sc.pp.pca(adata, n_comps=50, use_highly_variable=True, random_state=RANDOM_STATE)


Found 13 leiden clusters: {'cluster_0': 436, 'cluster_1': 447, 'cluster_2': 299, 'cluster_3': 695, 'cluster_4': 159, 'cluster_5': 787, 'cluster_6': 1050, 'cluster_7': 432, 'cluster_8': 556, 'cluster_9': 594, 'cluster_10': 202, 'cluster_11': 378, 'cluster_12': 449}
Built contrast matrix with 24 contrasts (1 pairwise + 13 global + 10 on ST8059048)
Running differential expression...


...

  ✓ activity X_activity_mlm_progeny: 6484 spots x 14 sets


  ✓ activity X_activity_ulm_collectri: 6484 spots x 718 sets


  ✓ activity X_activity_aucell_hallmark: 6484 spots x 50 sets


/home/klaus/ws/zarr-test-datasets/.claude/worktrees/enrichment-subset-comparisons-9a20d0/.venv/lib/python3.12/site-packages/ome_zarr/writer.py:819: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da.to_zarr(


Wrote SpatialData store -> test-data/visium-brain-de-enrichment-test-data-format.zarr

Wrote 24 DE contrasts, 73 enrichments and 3 activity matrices to test-data/visium-brain-de-enrichment-test-data-format.zarr
